# S4 · Data Cleaning — step-by-step walkthrough

**Raw data is not training data.** A web crawl becomes a corpus only after it is
normalized, filtered, deduplicated, decontaminated, and stamped with provenance —
**and every one of those steps must be script-aware, or it silently deletes the
Indic text it was supposed to keep.**

This notebook runs the *same functions* as [`clean.py`](clean.py) (we `import` them, we
don't re-type them — so this notebook can never drift from the real pipeline), one stage
at a time, on a **small slice** so it finishes in under a minute. The real run processes
**60,000 docs (~19 min)**; its true numbers are loaded at the very end from `out/stats.json`.

Dataset: **AI4Bharat / Sangraha**, split `verified/hin` — a verified Hindi web+PDF crawl.

### The 8 stages
| # | stage | script-aware? |
|---|-------|:---:|
| 1 | Normalization | ● |
| 2 | Format discipline (ghost-tags) | |
| 3 | Language ID & validation | ● |
| 4 | Quality filtering | ● |
| 5 | PII removal | ● |
| 6 | Deduplication (local vs global) | |
| 7 | Decontamination | |
| 8 | Manifest / provenance | |

The dots mark where an English-tuned cleaner would erase good Hindi. Stage 4 *measures* that erasure.

> **How to run:** pick the `.venv` Python kernel (top-right), then run cells top to bottom.

## Setup — import the real pipeline

We add the stage folder to the path and `import clean`. Importing only defines the functions/regexes (config, no side effects) — it does **not** run the 19-min pipeline.

In [ ]:
import sys, json, hashlib, time
from pathlib import Path

HERE = Path.cwd()
if not (HERE / 'clean.py').exists() and (HERE / 's04-data-cleaning' / 'clean.py').exists():
    HERE = HERE / 's04-data-cleaning'          # ran from repo root instead of the stage folder
sys.path.insert(0, str(HERE))

import clean          # <- the actual pipeline; every stage below calls these functions
print('loaded clean.py from', HERE)

### Load a small real slice

The real run uses 60,000 docs. We read just the first **1,000** here for speed. If the source parquet has been cleaned from scratch, we fall back to a tiny baked-in sample so every cell below still runs.

In [ ]:
import pyarrow.parquet as pq

N = 1000   # teaching slice; real run = 60,000

def load_slice(n=N):
    if clean.RAW.exists():
        d = next(pq.ParquetFile(clean.RAW).iter_batches(batch_size=n)).to_pydict()
        return [{'id': d['doc_id'][i], 'text': d['text'][i], 'type': d['type'][i]}
                for i in range(len(d['text']))]
    print('RAW parquet not found — using a tiny baked-in Hindi sample.')
    return [{'id': f'sample-{i}', 'type': 'web', 'text': t} for i, t in enumerate([
        'यह एक सामान्य हिंदी वाक्य है जिसमें पर्याप्त शब्द हैं। ' * 6,
        'भारत एक विशाल देश है और यहाँ अनेक भाषाएँ बोली जाती हैं। ' * 5])]

docs = load_slice()
print(f'{len(docs)} docs |',
      {k: sum(x['type'] == k for x in docs) for k in {d['type'] for d in docs}})
print('\n--- first doc (truncated) ---\n', docs[0]['text'][:280])

## Counting tokens honestly

We size every budget in **tokens**, so *how* we count matters. The naive `chars / 4` rule
is tuned for English and is **wrong for Indic by several times** — it corrupts every
downstream number. We count with the real **sarvam-1** Indic tokenizer (68k vocab) instead.

Sangraha Hindi lands at a *fertility* of ~**3.5 chars/token**; the naive rule assumes 4.

In [ ]:
tok = clean.load_tokenizer()          # sarvam-1; cached after first download

sample = docs[0]['text']
real  = len(tok.encode(sample).ids)
naive = len(sample) // 4              # the chars/4 estimate §11 warns about
print(f'chars={len(sample)}  real sarvam tokens={real}  naive chars/4={naive}')
print(f'fertility = {len(sample)/real:.2f} chars/token  (naive rule assumes 4.0)')

def total_tokens(ds):
    return sum(clean.tok_counts(tok, [d['text'] for d in ds]))

base_docs, base_tokens = len(docs), total_tokens(docs)
print(f'\nbaseline slice: {base_docs} docs, {base_tokens:,} tokens')

## Stage 1 · Normalization  ●

**What:** put every doc in one canonical form. **How:** Unicode NFC → strip invisible
control/noise chars → unescape HTML entities (`&amp;`→`&`) → collapse whitespace.
**Why:** a byte-level tokenizer otherwise wastes vocabulary on zero-width spaces and broken
byte fragments (V4 shipped 46 garbage tokens for exactly this).

**Indic fix — the sovereign corner:** we **keep** the zero-width joiner (ZWJ) and
non-joiner (ZWNJ) — they are *real letters* in Brahmic scripts — while stripping true noise
(ZWSP, BOM, bidi overrides, replacement char). A cleaner that strips *all* invisibles mangles Hindi.

In [ ]:
# hand example: letter + ZWNJ(keep) + letter + &amp; + ZWSP(drop) + BOM(drop)
ex = "अ‌आ&amp;​﻿"
out, garbage = clean.normalize(ex)
print('before:', repr(ex))
print('after :', repr(out))
print('kept ZWNJ?', '‌' in out,
      '| dropped ZWSP/BOM?', '​' not in out and '﻿' not in out,
      '| unescaped &amp;?', '&' in out, '| garbage removed:', garbage)

# apply to the whole slice
touched = 0
for d in docs:
    new, _ = clean.normalize(d['text'])
    touched += (new != d['text'])
    d['text'] = new
after = total_tokens(docs)
print(f'\nslice: {touched}/{len(docs)} docs changed; {base_tokens-after:,} tokens reclaimed')

## Stage 2 · Format discipline (ghost-tags)

**What:** stop fake conversation structure leaking into pretraining. **How:** regex-detect
literal markers from mixed sources (`<|user|>`, `[INST]`, `### Human:`, `<s>`…) and rewrite them.
**Why:** if pretraining learns `[INST]` as ordinary subwords, the model later *fights* the
tokenizer's real special tokens during fine-tuning (a priority-0 V4 bug). Rare in verified
web text — shown here so you can see the mechanism.

In [ ]:
ex = "नमस्ते <|user|> यह टेक्स्ट है [INST] और ### Human: लीक हो गया </s>"
print('markers found:', [m for g in clean.GHOST.findall(ex) for m in ([g] if isinstance(g,str) else g) if m])
print('rewritten    :', clean.GHOST.sub(' ', ex))

gt = 0
for d in docs:
    if clean.GHOST.search(d['text']):
        gt += 1; d['text'] = clean.GHOST.sub(' ', d['text'])
print(f'\nslice: {gt} docs carried leaked markers')

## Stage 3 · Language ID & validation  ●

**What:** confirm each doc is actually Hindi — *don't trust the folder name.* **How:**
compute the Devanagari character ratio at runtime; drop docs below 50%. **Why:** web crawls
are mislabeled often enough that trusting the folder pollutes per-language pools and skews the
fertility numbers we size budgets with (where V4's silent Telugu language-code bug lived).

In [ ]:
for s in ["यह पूरी तरह हिंदी वाक्य है।",
          "this is entirely english text here",
          "हिंदी and English mixed आधा आधा text वाक्य"]:
    print(f'{clean.deva_ratio(s):.2f}  {s}')

kept, dropped, ex_drop = [], 0, None
for d in docs:
    if clean.deva_ratio(d['text']) >= 0.50:
        kept.append(d)
    else:
        dropped += 1
        if ex_drop is None: ex_drop = d['text'][:140]
docs = kept
print(f'\nslice: dropped {dropped} non-Hindi docs (folder said hin, script disagreed)')
if ex_drop: print('example dropped:', repr(ex_drop))

## Stage 4 · Quality filtering  ●  — and the *Indic-erasure* measurement

**What:** decide if a doc is worth keeping. **How:** heuristic rules — min length, mean word
length, symbol-to-word ratio, duplicate-line fraction, terminal-punctuation rate.

**Indic fix:** thresholds are **script-aware** — the danda `।` counts as sentence-ending
punctuation, and we **drop** the English stop-word rule (which good Hindi always fails). We
*also* run the English-tuned filter — not to use it, but to **measure** how much real Hindi it
would wrongly delete. That measured number is the "other concern" of the whole session.

In [ ]:
# same good Hindi doc, judged by both filters
hindi = ("यह एक सामान्य हिंदी वाक्य है जिसमें पर्याप्त शब्द हैं। " * 4).strip()
print('script-aware keeps it? ', clean.quality_ok(hindi, script_aware=True))
print('english-tuned keeps it?', clean.quality_ok(hindi, script_aware=False),
      ' <- the EN filter deletes good Hindi. That is Indic erasure.')

kept, sa_drop, en_would, erasure = [], 0, 0, 0; reasons = {}
for d in docs:
    ok_sa, why = clean.quality_ok(d['text'], True)
    ok_en, _   = clean.quality_ok(d['text'], False)
    en_would += (not ok_en)
    if not ok_sa:
        sa_drop += 1; reasons[why] = reasons.get(why, 0) + 1
    else:
        kept.append(d)
        erasure += (not ok_en)          # good docs we kept that the EN filter would have killed
docs = kept
print(f'\nslice: script-aware dropped {sa_drop} {reasons}')
print(f'english-tuned WOULD drop {en_would}; of the docs we KEPT, {erasure} good Hindi docs '
      f'the English filter would have erased.')

## Stage 5 · PII removal  ●

**What:** mask personal data — for people, and for the corpus's legal usability. **How:** a
regex layer masks structured identifiers: emails, Indian mobile numbers, IPs, Aadhaar-shaped
numbers. (A production ML *name-layer* is the second pass — described in the design doc, not
run here; for Indic names it is sharper because a common name is often also a place.)

In [ ]:
ex = "संपर्क करें a@b.com या 9876543210, IP 192.168.0.1, आधार 1234 5678 9012"
masked, counts = clean.scrub_pii(ex)
print(masked, '\n', counts)

pii, pii_docs = {}, 0
for d in docs:
    new, c = clean.scrub_pii(d['text'])
    if c:
        pii_docs += 1
        for k, v in c.items(): pii[k] = pii.get(k, 0) + v
        d['text'] = new
print(f'\nslice: masked spans in {pii_docs} docs: {pii}')

## Stage 6 · Deduplication — local vs **global**

**What:** remove repeated learning signal. **How:** exact-hash dedup, then near-dup via
**shingles → MinHash → LSH banding**. A doc becomes a set of overlapping k-word shingles;
MinHash squeezes that set into a short signature; LSH buckets similar signatures so near-dups
collide without an all-pairs comparison.

**The point:** duplication is **global**. We split the slice into 4 shards and dedup each
*locally*, then run one *global* pass — the gap is the cross-shard duplicates that two
"individually clean" shards share but neither local pass could ever see. That gap is why a
real run needs one large-memory machine holding the whole index.

In [ ]:
import numpy as np
# toy: two identical docs + one different -> LSH flags exactly one as a dup
toy = ["the quick brown fox jumps over the lazy dog again and again"] * 2 + \
      ["completely unrelated sentence with different words entirely here now"]
print('flagged near-dup indices:', clean.lsh_dupes(clean.signatures(toy), [0, 1, 2]), '(expect {1})')

sigs = clean.signatures([d['text'] for d in docs])      # ~15-20s, pure Python by design
n, K = len(docs), 4
shard = np.arange(n) % K
local = set()
for s in range(K):
    local |= clean.lsh_dupes(sigs, [i for i in range(n) if shard[i] == s])
glob = clean.lsh_dupes(sigs, list(range(n)))
print(f'\nlocal (4 shards): {len(local)} near-dups | global: {len(glob)} | '
      f'cross-shard MISSED by local: {len(glob - local)}')
docs = [d for i, d in enumerate(docs) if i not in glob]
print('remaining docs:', len(docs))

## Stage 7 · Decontamination

**What:** keep the firewall between training data and benchmarks. **How:** fingerprint the
eval sets, scan every shard for overlap, remove any doc carrying a test item, and plant a
**canary string** to detect future leaks. **Why:** once a test item trains the model, its
score is no longer real. We *inject* a contaminated doc and watch the scan catch it.

In [ ]:
eval_strings = [d['text'][:120] for d in docs[:3]]      # pretend these are held-out eval items
eval_fp = {hashlib.sha1(s.encode()).hexdigest() for s in eval_strings}
CANARY = "CANARY-a4b7-do-not-train-9f2e"

def contaminated(d):
    return any(hashlib.sha1(d['text'][i:i+120].encode()).hexdigest() in eval_fp
               for i in range(0, max(1, len(d['text']) - 120), 40))

victim = dict(docs[20]); victim['text'] = eval_strings[0] + " " + victim['text']
print('planted eval item caught in victim doc?', contaminated(victim))
print('canary string detectable?', CANARY in (docs[21]['text'] + " " + CANARY))

before = len(docs)
docs = [d for d in docs if not contaminated(d)]
print(f'removed {before - len(docs)} docs overlapping the (pretend) eval set')

## Stage 8 · Manifest / provenance

**What:** make the corpus reproducible and auditable. **How:** a per-shard record — source,
license, contributor, **hash of the cleaning script**, **hash of the cleaned content**, token
count, language breakdown. Same input → same output → **same hash**. A missing/unsafe license
stamps the shard **BLOCKED**; that gate is what a shard must pass to enter the corpus.

In [ ]:
clean_text = "\n".join(d['text'] for d in docs)
manifest = {
    'source': clean.SOURCE, 'license': clean.LICENSE, 'contributor': clean.CONTRIBUTOR,
    'cleaning_script_sha256': hashlib.sha256(Path(clean.__file__).read_bytes()).hexdigest()[:16] + '…',
    'content_sha256': hashlib.sha256(clean_text.encode()).hexdigest(),
    'doc_count': len(docs), 'language': {'hin_Deva': 1.0},
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
blocked = manifest['license'].lower() in {'unknown', 'unsafe', ''} or not all(
    manifest.get(k) for k in ['source', 'license', 'content_sha256'])
manifest['status'] = 'BLOCKED' if blocked else 'ALLOWED'
print(json.dumps(manifest, ensure_ascii=False, indent=2))

# reproducibility check: same text -> same content hash
assert hashlib.sha256(clean_text.encode()).hexdigest() == manifest['content_sha256']
print('\nsame text -> same content hash  ✓')

## The real run — full 60,000-doc numbers

Everything above ran on a 1,000-doc slice so it would finish fast. The production run
(`.venv/Scripts/python clean.py`, ~19 min) wrote its true funnel to `out/stats.json`. Here it is —
the actual docs/tokens surviving each stage, and the naive-vs-real token gap the manifest records.

In [ ]:
stats = json.loads((HERE / 'out' / 'stats.json').read_text(encoding='utf-8'))
print(f"retained {stats['retained_docs_pct']}% docs, {stats['retained_tokens_pct']}% tokens | "
      f"fertility {stats['fertility_chars_per_token']} chars/token\n")
print(f"{'stage':<13}{'docs':>9}{'tokens':>13}{'-docs':>8}{'-tokens':>11}")
for f in stats['funnel']:
    print(f"{f['stage']:<13}{f['docs']:>9,}{f['tokens']:>13,}{f['removed_docs']:>8,}{f['removed_tokens']:>11,}")
print(f"\nnaive chars/4 estimate: {stats['naive_token_estimate']:,} tokens  vs  "
      f"real {stats['final']['tokens']:,}  — the gap §11 warns about.")

## Recap

1. **Normalize** – NFC + strip noise, *keep ZWJ/ZWNJ*.
2. **Ghost-tags** – rewrite leaked `<|user|>`/`[INST]` markers.
3. **Language ID** – Devanagari ratio ≥ 50%, don't trust the folder.
4. **Quality** – script-aware heuristics; *measure* what an English filter would erase.
5. **PII** – regex-mask emails/phones/IPs/Aadhaar.
6. **Dedup** – shingles→MinHash→LSH; global catches what local shards miss.
7. **Decontaminate** – fingerprint eval sets, drop overlaps, plant a canary.
8. **Manifest** – hash script + content; license gate → ALLOWED/BLOCKED.

The through-line: **every script-aware step (●) exists because an English-tuned cleaner
would delete good Hindi.** Stage 4 turns that risk into a number.